# Clustering Comparison: Wine Dataset

Comparing K-Means++, DBSCAN, HDBSCAN, Gaussian Mixture Models, and Spectral
Clustering on `sklearn.datasets.load_wine()` (178 samples, 13 chemical-analysis
features, 3 cultivar classes as ground truth). Since ground truth exists, both
internal (unsupervised) and external (supervised) validity metrics are computed.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sc4020 import generate_figures as gf
from sc4020 import methods, tuning
from sc4020.datasets import load_wine
from sc4020.paths import results_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
METRIC_COLS = [
    "method", "fit_seconds", "n_clusters", "noise_fraction", "largest_cluster_share",
    "silhouette", "davies_bouldin", "calinski_harabasz",
]
EXTERNAL_COLS = ["ari", "nmi", "ami", "fmi"]

In [ ]:
ds = load_wine()
print(f"{ds.name}: X={ds.data.shape}, classes={ds.target_names}, n_true_clusters={ds.n_true_clusters}")

## Exploratory look

In [ ]:
df_raw = pd.DataFrame(ds.data, columns=ds.feature_names)
df_raw["target"] = [ds.target_names[i] for i in ds.target]
display(df_raw.describe().T)
df_raw["target"].value_counts()

## 2D embedding for visualization

Clustering itself runs on the full standardized 13-D feature space; PCA to 2D
here is *only* used to plot the results.

In [ ]:
X_2d, xlabel, ylabel = gf.project_2d(ds.data)

## DBSCAN eps selection

`eps` is chosen from the knee of the sorted k-distance graph (k = min_samples,
a `2 * n_features` heuristic), then fine-tuned by a small silhouette-scored
grid search inside `tuning.select_dbscan`. Shown here for the default
`min_samples` for illustration.

In [ ]:
ms = tuning.default_min_samples(ds.data.shape[1])
curve = tuning.k_distance_curve(ds.data, ms)
knee_idx, eps_at_knee = tuning.knee(curve)
gf.plot_k_distance(curve, knee_idx, eps_at_knee, min_samples=ms)
plt.show()

## Fit all 5 methods

K for K-Means++ / GMM / Spectral Clustering is set to the true number of
classes (3) for a fair comparison against DBSCAN/HDBSCAN's auto-discovered
cluster count. Spectral Clustering uses an RBF affinity (wine's clusters are
roughly convex/globular, not manifold-shaped).

In [ ]:
# The shared rule (tuning.select_all): K-Means k by silhouette (the number of classes when known),
# GMM by BIC, DBSCAN and HDBSCAN by the best silhouette with >= 2 clusters and <= 50% noise.
# Spectral clustering runs at the K-Means k.
selections = tuning.select_all(ds.data, k=ds.n_true_clusters, y_true=ds.target)
params = {m: s.params for m, s in selections.items()}
params["Spectral"] = {"n_clusters": params["K-Means"]["n_clusters"], "affinity": "rbf"}
results = methods.run_models(methods.build_models(params), ds.data, params=params)
for name, r in results.items():
    print(f"{name:22s} params={r.params}  fit_seconds={r.fit_seconds:.3f}")

## Metrics

In [ ]:
results_df = gf.build_results_table(ds.name, ds.data, results, ds.target)
results_df.to_csv(results_dir("datasets") / "wine_tuned_methods_results.csv", index=False)
display(results_df[METRIC_COLS])
display(results_df[["method"] + EXTERNAL_COLS])

## Cluster visualizations vs. ground truth

In [ ]:
fig = gf.plot_cluster_grid(ds.data, results, y_true=ds.target, class_names=ds.target_names,
                           title="Wine: predicted clusters vs. ground truth (PCA 2D)")
plt.show()

## Discussion

- **K-Means++ and GMM** match the ground truth closely (high ARI/NMI): wine's
  3 cultivar classes are close to convex, similarly-sized Gaussian blobs in
  the standardized feature space, exactly the assumption these two methods make.
- **Spectral Clustering** (RBF affinity) is competitive but more sensitive to
  the affinity bandwidth than the K-based methods here.
- **DBSCAN and HDBSCAN underperform** and typically collapse to fewer clusters
  with a large noise fraction — a known weakness of density-based methods in
  moderate-dimensional (13-D) spaces, where distances concentrate and density
  differences between the true classes become harder to separate with a single
  global `eps`/`min_cluster_size`.
- This is a case where knowing the true K (used here) gives K-Means++/GMM/Spectral
  a real advantage; DBSCAN/HDBSCAN had to discover the cluster count unsupervised.